In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run

/Workspace/Users/donprabhash0071@gmail.com/databricks/fmcg/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# Just for example, we are not using this.
# we can write/dropdown the catalog name in the widget without making change in code.
# The file path can directly use value in widget as directory to load a file.
#  
dbutils.widgets.text('catalog', 'fmcg', 'Catalog Name')
dbutils.widgets.text('source', 'customers', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
source = dbutils.widgets.get('source')

In [0]:
print(catalog, source)

In [0]:
df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('/Volumes/fmcg/fmcg_data/data/2_child_company/full_load/gross_price/gross_price.csv') \
    .withColumn('read_timestamp', F.current_timestamp()) \
    .select('*', '_metadata.file_name', '_metadata.file_size')
    # filename & filezise is used to track the source of the data and data lineage

display(df.limit(10))

In [0]:
df.write.format('Delta') \
    .mode('overwrite') \
    .option('enableChangeDataFeed', 'true') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{source}')